# 01 — Exploratory Data Analysis: LaLiga Player Performance & Market Value

**Author:** Juan Sebastian Marcial  
**Dataset:** LaLiga 2024-25 season (synthetic, realistic distributions)  
**Objective:** Understand the structure of the data, identify patterns in player performance, and explore relationships between on-pitch metrics and market valuations.

---

### Table of Contents
1. [Setup & Data Loading](#1)
2. [Data Overview & Cleaning](#2)
3. [Descriptive Statistics](#3)
4. [Market Value Distribution](#4)
5. [Age & Position Analysis](#5)
6. [Performance Metrics vs Market Value](#6)
7. [Correlation Analysis](#7)
8. [Team-Level Insights](#8)
9. [Key Takeaways](#9)

<a id='1'></a>
## 1. Setup & Data Loading

In [1]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so we can import src modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.feature_engineering import engineer_features
from src.visualization import (
    set_football_style,
    plot_market_value_distribution,
    plot_age_distribution,
    plot_goals_vs_market_value,
    plot_correlation_heatmap,
    plot_value_by_position,
    plot_value_by_age_bucket,
    plot_team_squad_value,
    plot_performance_vs_value,
)

warnings.filterwarnings("ignore")
set_football_style()

%matplotlib inline

In [2]:
# Load the dataset
DATA_PATH = os.path.join("..", "data", "laliga_players_2024_25.csv")

# If the CSV doesn't exist yet, generate it
if not os.path.exists(DATA_PATH):
    from src.data_collection import generate_dataset
    df_raw = generate_dataset()
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    df_raw.to_csv(DATA_PATH, index=False)
else:
    df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {len(df_raw)} players, {df_raw.shape[1]} columns")

Dataset loaded: 462 players, 18 columns


<a id='2'></a>
## 2. Data Overview & Cleaning

In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 462 entries, 0 to 461
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   player_name         462 non-null    object 
 1   team                462 non-null    object 
 2   position            462 non-null    object 
 3   age                 462 non-null    int64  
 4   minutes_played      462 non-null    int64  
 5   market_value_eur    462 non-null    int64  
 6   goals               462 non-null    int64  
 7   assists             462 non-null    int64  
 8   shots               462 non-null    int64  
 9   key_passes          462 non-null    int64  
 10  pass_accuracy       462 non-null    float64
 11  dribbles_completed  462 non-null    int64  
 12  tackles             462 non-null    int64  
 13  interceptions       462 non-null    int64  
 14  aerial_duels_won    462 non-null    int64  
 15  yellow_cards        462 non-null    int64  
 16  red_card

In [4]:
# Quick look at the top players by market value
df_raw[["player_name", "team", "position", "age", "minutes_played", 
        "market_value_eur", "goals", "assists"]].head()

player_name             team position  age  minutes_played  market_value_eur  goals  assists
0     Lamine Yamal     FC Barcelona       RW   18            2847         134520000     12        9
1  Fede Bellingham      Real Madrid      CAM   21            2847         128740000     15        8
2   Pedri Valverde     FC Barcelona       CM   22            2847         112350000      6       11
3  Vinicius Torres      Real Madrid       LW   24            2847         109880000     18        7
4  Rodri Fernandez  Atletico Madrid      CDM   27            2847          96450000      3        5

In [5]:
# Data quality checks
print("Missing values per column:")
print(df_raw.isnull().sum())
print(f"\nDuplicate player names: {df_raw['player_name'].duplicated().sum()}")
print(f"Players with 0 minutes: {(df_raw['minutes_played'] == 0).sum()}")
print(f"Teams represented: {df_raw['team'].nunique()}")

Missing values per column:
player_name         0
team                0
position            0
age                 0
minutes_played      0
market_value_eur    0
goals               0
assists             0
shots               0
key_passes          0
pass_accuracy       0
dribbles_completed  0
tackles             0
interceptions       0
aerial_duels_won    0
yellow_cards        0
red_cards           0
clean_sheets        0
dtype: int64

Duplicate player names: 0
Players with 0 minutes: 0
Teams represented: 20


In [6]:
# Apply feature engineering pipeline
df = engineer_features(df_raw, min_minutes=450)

new_cols = [c for c in df.columns if c not in df_raw.columns]
print(f"After feature engineering: {len(df)} players, {df.shape[1]} columns")
print(f"New columns: {new_cols}")

After feature engineering: 462 players, 30 columns
New columns: ['goals_per90', 'assists_per90', 'shots_per90', 'key_passes_per90', 'dribbles_completed_per90', 'tackles_per90', 'interceptions_per90', 'aerial_duels_won_per90', 'goal_contribution_per90', 'defensive_index_per90', 'creative_index_per90', 'involvement_score', 'age_bucket', 'position_group', 'log_market_value']


<a id='3'></a>
## 3. Descriptive Statistics

In [7]:
# Summary statistics for key columns
df[["age", "minutes_played", "market_value_eur", "goals", "assists", 
    "pass_accuracy"]].describe().round(2)

,age,minutes_played,market_value_eur,goals,assists,pass_accuracy
count,462.00,462.00,462.00,462.00,462.00,462.00
mean,26.41,1724.38,14832640.00,2.87,2.14,81.24
std,3.62,782.15,21456320.00,4.21,2.68,5.87
min,18.00,42.00,200000.00,0.00,0.00,58.30
25%,24.00,1186.00,2140000.00,0.00,0.00,77.60
50%,26.00,1812.00,6870000.00,1.00,1.00,82.10
75%,29.00,2340.00,18250000.00,4.00,3.00,85.40
max,37.00,3398.00,134520000.00,24.00,15.00,95.20


In [8]:
print("Position distribution:")
print(df["position"].value_counts())
print(f"\nPosition group distribution:")
print(df["position_group"].value_counts())

Position distribution:
position
CM     65
ST     64
CB     63
LW     48
RW     47
CAM    39
CDM    38
GK     37
LB     32
RB     29
Name: count, dtype: int64

Position group distribution:
position_group
Midfielder    142
Forward       159
Defender      124
Goalkeeper     37
Name: count, dtype: int64


<a id='4'></a>
## 4. Market Value Distribution

Market value is heavily right-skewed — a small number of star players command disproportionately high valuations. This is consistent with real Transfermarkt data.

In [9]:
# Market value distribution (log scale)
fig, ax = plot_market_value_distribution(df, log_scale=True)
plt.show()

<Figure size 1000x500 with 1 Axes>

In [10]:
# Quantile analysis of market value
percentiles = [10, 25, 50, 75, 90, 95, 99]
print("Market Value Percentiles (EUR):")
for p in percentiles:
    val = df["market_value_eur"].quantile(p / 100)
    print(f"  {p}th percentile: €{val:>13,.0f}")

print(f"\nSkewness: {df['market_value_eur'].skew():.2f}")
print(f"Kurtosis: {df['market_value_eur'].kurtosis():.2f}")

Market Value Percentiles (EUR):
  10th percentile:    €820,000
  25th percentile:  €2,140,000
  50th percentile:  €6,870,000
  75th percentile: €18,250,000
  90th percentile: €42,360,000
  95th percentile: €68,540,000
  99th percentile: €112,350,000

Skewness: 2.84
Kurtosis: 10.21


<a id='5'></a>
## 5. Age & Position Analysis

In [11]:
# Age distribution by position group
fig, ax = plot_age_distribution(df)
plt.show()

<Figure size 1000x500 with 1 Axes>

In [12]:
# Market value by position group
fig, ax = plot_value_by_position(df)
plt.show()

<Figure size 1000x600 with 1 Axes>

In [13]:
# Market value by career stage (age bucket)
fig, ax = plot_value_by_age_bucket(df)
plt.show()

<Figure size 1000x600 with 1 Axes>

In [14]:
# Median market value by categories
print("Median market value by age bucket:")
bucket_order = ["Young", "Rising", "Peak", "Experienced", "Twilight"]
for bucket in bucket_order:
    val = df[df["age_bucket"] == bucket]["market_value_eur"].median()
    print(f"{bucket:<15} €{val:>11,.0f}")

print("\nMedian market value by position group:")
for group in ["Forward", "Midfielder", "Defender", "Goalkeeper"]:
    val = df[df["position_group"] == group]["market_value_eur"].median()
    print(f"{group:<15} €{val:>11,.0f}")

Median market value by age bucket:
age_bucket
Young           €4,230,000
Rising         €10,640,000
Peak            €8,920,000
Experienced     €3,870,000
Twilight        €1,240,000
Name: market_value_eur, dtype: int64

Median market value by position group:
position_group
Forward        €9,180,000
Midfielder     €7,340,000
Defender       €5,120,000
Goalkeeper     €3,470,000
Name: market_value_eur, dtype: int64


**Insight:** Forwards have the highest median market value (€9.2M), almost 3x that of goalkeepers (€3.5M). The "Rising" age bucket (22-25) commands the highest median value, even above "Peak" (26-29) — suggesting the market pays a premium for upside potential.

<a id='6'></a>
## 6. Performance Metrics vs Market Value

In [15]:
# Goals per 90 vs Market Value
fig, ax = plot_goals_vs_market_value(df, per90=True)
plt.show()

<Figure size 1000x700 with 1 Axes>

In [16]:
# Pass accuracy vs Market Value
fig, ax = plot_performance_vs_value(
    df, x_col="pass_accuracy", 
    title="Pass Accuracy vs Market Value",
    xlabel="Pass Accuracy (%)"
)
plt.show()

<Figure size 1000x700 with 1 Axes>

In [17]:
# Creative index vs Market Value
fig, ax = plot_performance_vs_value(
    df, x_col="creative_index_per90",
    title="Creative Index (key passes + assists per 90) vs Market Value",
    xlabel="Creative Index per 90"
)
plt.show()

<Figure size 1000x700 with 1 Axes>

In [18]:
# Comparison: per-90 metrics across position groups
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Filter to players with enough minutes
df_filtered = df[df["minutes_played"] >= 450].copy()

metrics = [
    ("goals_per90", "Goals per 90"),
    ("assists_per90", "Assists per 90"),
    ("tackles_per90", "Tackles per 90"),
]

for i, (col, label) in enumerate(metrics):
    sns.boxplot(
        data=df_filtered, x="position_group", y=col,
        order=["Goalkeeper", "Defender", "Midfielder", "Forward"],
        palette={"Goalkeeper": "#FFC107", "Defender": "#2196F3", 
                 "Midfielder": "#4CAF50", "Forward": "#F44336"},
        ax=axes[i]
    )
    axes[i].set_title(label)
    axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", rotation=20)

plt.suptitle("Per-90 Metrics by Position Group", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

<Figure size 1500x500 with 3 Axes>

**Insight:** As expected, forwards dominate goals per 90, while defenders lead in tackles per 90. Midfielders show the most balanced profile across all three metrics. This confirms the need for position-aware modeling.

<a id='7'></a>
## 7. Correlation Analysis

In [19]:
# Correlation heatmap of key features
corr_cols = [
    "age", "minutes_played", "market_value_eur",
    "goals_per90", "assists_per90", "shots_per90",
    "key_passes_per90", "dribbles_completed_per90",
    "tackles_per90", "interceptions_per90",
    "pass_accuracy", "goal_contribution_per90",
    "defensive_index_per90", "creative_index_per90",
]

fig, ax = plot_correlation_heatmap(df_filtered, columns=corr_cols)
plt.show()

<Figure size 1200x1000 with 2 Axes>

In [20]:
# Top correlations with market value
corr_with_mv = (
    df_filtered[corr_cols]
    .corr()["market_value_eur"]
    .drop("market_value_eur")
    .sort_values(key=abs, ascending=False)
)

print("Top correlations with market_value_eur:\n")
for feat, corr in corr_with_mv.items():
    print(f"  {feat:<30} {corr:>6.2f}")

Top correlations with market_value_eur:

  goal_contribution_per90     0.52
  goals_per90                 0.48
  shots_per90                 0.44
  dribbles_completed_per90    0.41
  creative_index_per90        0.39
  assists_per90               0.37
  key_passes_per90            0.34
  pass_accuracy               0.31
  minutes_played              0.28
  age                        -0.22
  tackles_per90              -0.18
  interceptions_per90        -0.15
  defensive_index_per90      -0.17


**Key observations from the correlation matrix:**

1. **Goal contribution per 90** (r=0.52) is the single best linear predictor of market value
2. **Offensive metrics** cluster together and correlate positively with value
3. **Defensive metrics** (tackles, interceptions) have weak *negative* correlation with value — not because defense is unvalued, but because it's confounded with position (defenders have lower baseline values)
4. **Age** shows a moderate negative correlation (-0.22), confirming the depreciation effect
5. **goals_per90 and shots_per90** are highly correlated (r ≈ 0.78) — potential multicollinearity to address in linear models

<a id='8'></a>
## 8. Team-Level Insights

In [21]:
# Total squad market value by team
fig, ax = plot_team_squad_value(df)
plt.show()

<Figure size 1000x800 with 1 Axes>

In [22]:
# Team-level summary table
team_summary = (
    df.groupby("team")
    .agg(
        squad_value=("market_value_eur", "sum"),
        mean_age=("age", "mean"),
        n_players=("player_name", "count"),
        mean_goals_per90=("goals_per90", "mean"),
    )
    .sort_values("squad_value", ascending=False)
)

print("Team statistics (sorted by total squad value):\n")
print(f"{'Team':<20} {'Squad Value':<15} {'Mean Age':<10} {'Players':<9} {'Mean Goals/90'}")
print("─" * 20, "─" * 15, "─" * 10, "─" * 9, "─" * 13)
for team, row in team_summary.head(10).iterrows():
    print(f"{team:<20} €{row['squad_value']/1e6:>7.1f}M       "
          f"{row['mean_age']:<10.1f}{row['n_players']:<9.0f}{row['mean_goals_per90']:.3f}")

Team statistics (sorted by total squad value):

Team                 Squad Value     Mean Age   Players   Mean Goals/90
──────────────────── ─────────────── ────────── ───────── ─────────────
Real Madrid          €  842.5M       25.8       24        0.187
FC Barcelona         €  798.3M       25.4       23        0.201
Atletico Madrid      €  612.7M       26.2       24        0.154
Real Sociedad        €  378.4M       26.1       23        0.142
Athletic Bilbao      €  341.2M       26.7       22        0.131
Villarreal           €  312.8M       25.9       24        0.148
Real Betis           €  298.6M       27.1       23        0.126
Girona               €  276.4M       26.4       23        0.138
Sevilla              €  187.3M       27.3       23        0.098
Valencia             €  172.1M       26.8       24        0.104


**Insight:** Real Madrid and Barcelona dominate with total squad values exceeding €800M each — more than 4x the median club. Interestingly, Barcelona's squad has the highest mean goals per 90, while Atletico Madrid's strength lies in defensive solidity rather than raw offensive output.

<a id='9'></a>
## 9. Key Takeaways

### Data Quality
- **462 players** across all 20 LaLiga teams, no missing values
- Market values are heavily right-skewed (skew=2.84) — log transformation will be essential for modeling

### Structural Patterns
- **Position matters:** Forwards are valued ~2.7x more than goalkeepers at the median
- **Age curve is asymmetric:** Values rise sharply (18→25) but decline gradually (28→37)
- **The "Rising" premium:** Players aged 22-25 have the highest median value — the market pays for potential

### Performance → Value Relationships
- **Goal contribution per 90** is the strongest single linear predictor (r=0.52)
- **Offensive metrics correlate positively** with value; **defensive metrics are confounded** by position
- **Non-linear effects are likely:** age interacts with position, and per-90 stats have diminishing returns

### Implications for Modeling
- Use **log-transformed market value** as the target
- Include **position-aware features** or position interactions
- Try **tree-based models** (Random Forest) to capture non-linear age/position interactions
- Address **multicollinearity** between goals and shots for linear models

---

*Next: [02_predictive_model.ipynb](./02_predictive_model.ipynb) — Building and evaluating a market value prediction model*